# 4. Inheritance

Inheritance in Python looks similar to Java, but with one massive difference: Python supports **multiple inheritance**. Java limits you to single class inheritance + interfaces; Python has no such restriction.

This notebook covers: basic inheritance, `super()`, method overriding, multiple inheritance, MRO (Method Resolution Order), type checking, `__init_subclass__`, and preventing inheritance.

### 4.1 Basic Inheritance & `super()`

**☕ JAVA:**
```java
public class Dog extends Animal {
    public Dog(String name, String breed) {
        super(name);       // Call parent constructor
        this.breed = breed;
    }
}
```

**🐍 PYTHON:** Use `class Dog(Animal):` instead of `extends`. Same `super()` concept, but no need to pass `self` to `super()` (Python 3 magic).

In [1]:
class Animal:
    def __init__(self, name: str):
        self.name = name
    
    def speak(self) -> str:
        return "Some sound"
    
    def __str__(self):
        return f"{self.__class__.__name__}({self.name})"
    
class Dog(Animal):
    def __init__(self, name: str, bread: str):
        super().__init__(name)
        self.bread = bread

dog = Dog("Buddy", "Labrador")
print(f"Name: {dog.name}")
print(f"Bread: {dog.bread}")
print(f"Speak: {dog.speak()}")
print(dog)

Name: Buddy
Bread: Labrador
Speak: Some sound
Dog(Buddy)


### 4.2 Method Overriding

**☕ JAVA:** `@Override` annotation is optional but recommended.

**🐍 PYTHON:** No `@Override` annotation exists. Just define a method with the same name — it automatically overrides. You can call `super().method()` to invoke the parent's version.

In [3]:
class Animal:
    def __init__(self, name: str):
        self.name = name
    
    def speak(self) -> str:
        return "..."
    
    def describe(self) -> str:
        return f"{self.name} says {self.speak()}"

class Dog(Animal):
    def speak(self): # Override - no annotation needed
        return "Woof! Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow Mewo"
    
class Parrot(Animal):
    def speak(self):
        return f"{super().speak()} Polly wants a carrot."

animals: list[Animal] = [Dog("Buddy"), Cat("Whisker"), Parrot("Polly")]
for animal in animals:
    print(f"{animal.describe()}")
    print(f"isinstance(animal, Animal): {isinstance(animal, Animal)}")
    print(f"type(animal): {type(animal)}")


Buddy says Woof! Woof!
isinstance(animal, Animal): True
type(animal): <class '__main__.Dog'>
Whisker says Meow Mewo
isinstance(animal, Animal): True
type(animal): <class '__main__.Cat'>
Polly says ... Polly wants a carrot.
isinstance(animal, Animal): True
type(animal): <class '__main__.Parrot'>


### When we want to check for inheritance then we use isinstance() no type()

In [8]:
my_dog, my_cat, my_parrot = Dog("Buddy"), Cat("Whisker"), Parrot("Polly") 

print(f"isinstance(my_dog, Animal): {isinstance(my_dog, Animal)}")
print(f"isinstance(my_dog, Dog): {isinstance(my_dog, Dog)}")
print(f"isinstance(my_dog, Cat): {isinstance(my_dog, Cat)}")

print(f"type(my_dog) is Dog: {type(my_dog) == Dog}")
print(f"type(my_dog) is Animal: {type(my_dog) == Animal}")

isinstance(my_dog, Animal): True
isinstance(my_dog, Dog): True
isinstance(my_dog, Cat): False
type(my_dog) is Dog: True
type(my_dog) is Animal: False


### 4.3 Multiple Inheritance — Java Can't Do This!

**☕ JAVA:** Only single class inheritance. Interfaces provide partial workaround:
```java
public class FlyingFish extends Fish implements Flyable { ... }
```

**🐍 PYTHON:** Full multiple inheritance — a class can inherit from **multiple** parent classes:
```python
class FlyingFish(Fish, Flyer): ...
```

In [9]:
class Swimmer:
    def swim(self) -> str:
        return "Swimming"

class Flyer:
    def fly(self) -> str:
        return "Flying"
    
class Runner:
    def run(self) -> str:
        return "Running"

class Duck(Swimmer, Flyer, Runner):
    def __init__(self, name: str):
        self.name = name

duck = Duck("Donald")
print(f"{duck.name} can:")
print(f"   {duck.swim()}")
print(f"   {duck.fly()}")
print(f"   {duck.run()}")

Donald can:
   Swimming
   Flying
   Running


### 4.4 MRO — Method Resolution Order

**☕ JAVA:** No need — single inheritance means there's only one path to resolve.

**🐍 PYTHON:** With multiple inheritance, the same method might exist in multiple parents. Python uses **C3 Linearization** to determine a consistent order. View it with `ClassName.__mro__` or `ClassName.mro()`.

The **Diamond Problem** occurs when class D inherits from B and C, which both inherit from A:

In [13]:
class A:
    def greet(self) -> str:
        return "Hello from A"

class B(A):
    def greet(self) -> str:
        return "Hello from B"

class C(A):
    def greet(self) -> str:
        return "Hello from C"
    
class D(B, C):
    pass

b = B()
print(b.greet())
c = C()
print(c.greet())
d = D()
print(d.greet())

# The manner that MRO functioning is:
print(f"MRO: {[cls.__name__ for cls in D.__mro__]}") # First searchs in class D after in B and so on

Hello from B
Hello from C
Hello from B
MRO: ['D', 'B', 'C', 'A', 'object']


> ⚠️ **Key insight:** `super()` doesn't always call the **direct parent** — it calls the **next class in the MRO**. This is why `B.greet()` calls `C.greet()` (not `A.greet()`) when called from `D`.

### 4.5 `isinstance()` vs `type()` — A Common Gotcha

**☕ JAVA:** `obj instanceof Dog` checks the full hierarchy.

**🐍 PYTHON:** `isinstance(obj, Dog)` and `issubclass(Dog, Animal)` both accept tuples for checking multiple types.

> ⚠️ **Gotcha:** `type(obj) == Class` checks the **exact** type only — it does NOT consider inheritance. Always prefer `isinstance()`.

In [ ]:
# The same exmple that had seen before
my_dog, my_cat, my_parrot = Dog("Buddy"), Cat("Whisker"), Parrot("Polly") 

print(f"isinstance(my_dog, Animal): {isinstance(my_dog, Animal)}")
print(f"isinstance(my_dog, Dog): {isinstance(my_dog, Dog)}")
print(f"isinstance(my_dog, Cat): {isinstance(my_dog, Cat)}")

print(f"type(my_dog) is Dog: {type(my_dog) == Dog}")
print(f"type(my_dog) is Animal: {type(my_dog) == Animal}")

### 4.6 `__init_subclass__` — Hook Into Subclassing

**☕ JAVA:** No equivalent — you can't execute code when a class is subclassed.

**🐍 PYTHON:** `__init_subclass__` is called automatically when a class is subclassed. Great for plugin registration, validation, or automatic setup.

In [1]:
# Auto-registration pattern — plugins register themselves!
class Plugin:
    """Base class that auto-registers all plugins."""
    _registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin._registry[cls.__name__] = cls
        print(f"  📦 Registered plugin: {cls.__name__}")

    @classmethod
    def get_plugins(cls) -> dict[str, type]:
        return dict(cls._registry)

# These register themselves automatically — no decorator or function call needed!
class CSVExporter(Plugin):
    def export(self, data): return "CSV..."

class JSONExporter(Plugin):
    def export(self, data): return "JSON..."

class XMLExporter(Plugin):
    def export(self, data): return "XML..."

print(f"\nAll plugins: {list(Plugin.get_plugins().keys())}")

  📦 Registered plugin: CSVExporter
  📦 Registered plugin: JSONExporter
  📦 Registered plugin: XMLExporter

All plugins: ['CSVExporter', 'JSONExporter', 'XMLExporter']


In [2]:
# Enforcing constraints on subclasses
class Serializable:
    """Base class that forces subclasses to define 'format_name'."""

    def __init_subclass__(cls, format_name: str = "", **kwargs):
        super().__init_subclass__(**kwargs)
        if not format_name:
            raise TypeError(f"{cls.__name__} must specify format_name")
        cls.format_name = format_name

class JsonDoc(Serializable, format_name="json"):
    pass

class XmlDoc(Serializable, format_name="xml"):
    pass

print(f"JsonDoc format: {JsonDoc.format_name}")
print(f"XmlDoc format:  {XmlDoc.format_name}")

try:
    class BadDoc(Serializable):   # No format_name!
        pass
except TypeError as e:
    print(f"\n❌ {e}")

JsonDoc format: json
XmlDoc format:  xml

❌ BadDoc must specify format_name


### 4.7 Preventing Inheritance — `@final`

**☕ JAVA:** `final class String { ... }` — cannot be subclassed.

**🐍 PYTHON:** Python 3.8+ provides `typing.final` for two purposes:

| Use | Java | Python |
|-----|------|--------|
| Prevent subclassing | `final class Foo` | `@final class Foo` |
| Prevent overriding | `final void method()` | `@final def method()` |

> ⚠️ `@final` is a **type-checker hint only** (enforced by mypy/pyright). Python itself won't stop subclassing at runtime. For runtime enforcement, combine with `__init_subclass__`.

In [14]:
from typing import final

class Base:
    @final
    def critical_method(self) -> str:
        return "Don't touch me!"
    
    def extensible_method(self) -> str:
        return "Override me"

class Child(Base):
    def critical_method(self):
        return "Do something else"
    
    def extensible_method(self):
        return "Everything is OK!"

c = Child()
print(f"Critical: {c.critical_method()}")
print(f"Extensible: {c.extensible_method()}")

Critical: Do something else
Extensible: Everything is OK!


In [15]:
class Dog():
    def speak(self): # Override - no annotation needed
        return "Woof! Woof!"

class Cat():
    def speak(self):
        return "Meow Mewo"
    
class Car():
    def sound(self):
        return "Vroum vroum"

def animal_speak(animal):
    print(animal.speak())

dog = Dog()
cat = Cat()
car = Car()

animal_speak(dog)
animal_speak(cat)
animal_speak(car)

Woof! Woof!
Meow Mewo


AttributeError: 'Car' object has no attribute 'speak'

---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Vehicle` base class with `make` and `year`. Create `Car(Vehicle)` and `Truck(Vehicle)` subclasses that override a `describe()` method.

In [18]:
# Exercise 1: Your code here
class Vehicle:

    def __init__(self, make: str, year: int ):
        self.make = make
        self.year = year

    def describe(self):
        return f"Made by {self.make} in {self.year}"

class Car(Vehicle):
    
    @property
    def describe(self):
        return "Car: " + super().describe()

class Truck(Vehicle):

    @property
    def describe(self):
        return  "Truck: " + super().describe()

In [19]:
car1 = Car("Audi", "2024")
truck1 = Truck("Scania", 2009)

print(car1.describe)
print(truck1.describe)

Car: Made by Audi in 2024
Truck: Made by Scania in 2009


**Exercise 2:** Create a diamond inheritance: `class ElectricAmphibious(ElectricVehicle, AmphibiousVehicle)` where both parents inherit from `Vehicle`. Print the MRO.

In [20]:
# Exercise 2: Your code here
class Vehicle:

    def __init__(self, make: str, year: int ):
        self.make = make
        self.year = year

    def describe(self):
        return f"Made by {self.make} in {self.year}"

class ElectricVehicle(Vehicle):
    
    def describe(self):
        return super().describe()

class AmphibiousVehicle(Vehicle):

    def describe(self):
        return super().describe()
    
class ElectricAmphibious(ElectricVehicle, AmphibiousVehicle):

    def describe(self):
        return super().describe()

In [26]:
ea1 = ElectricAmphibious("Toyota", 2020)
print(ea1.describe())

print(f"MRO: {[cls.__name__ for cls in ElectricAmphibious.mro()]}")

Made by Toyota in 2020
MRO: ['ElectricAmphibious', 'ElectricVehicle', 'AmphibiousVehicle', 'Vehicle', 'object']


**Exercise 3:** Create a `Handler` base class that auto-registers subclasses using `__init_subclass__`. Each subclass should specify a `handles` keyword (e.g., `class ImageHandler(Handler, handles='image')`). Add a class method `get_handler(media_type)` that returns the right handler.

In [27]:
# Exercise 3: Your code here
class Handler:
    _registry = {}
    
    def __init_subclass__(cls, handles: str, **kwargs):
        super().__init_subclass__(**kwargs)
        cls._registry[handles] = cls

    @classmethod    
    def get_handler(cls, media_type):
        return cls._registry.get(media_type)
    
class ImageHandler(Handler, handles='image'):
    pass

class VideoHandler(Handler, handles='video'):
    pass


In [30]:
Handler.get_handler("video")

__main__.VideoHandler

---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Inherit | `class Dog extends Animal` | `class Dog(Animal):` |
| Call parent constructor | `super(name)` | `super().__init__(name)` |
| Override | `@Override` annotation | Just redefine the method |
| Call parent method | `super.method()` | `super().method()` |
| Multiple inheritance | ❌ Not supported | ✅ `class D(B, C):` |
| Interfaces | `implements Runnable` | Not needed — use ABC or Protocol |
| Diamond problem | Can't happen | Solved by MRO (C3 linearization) |
| `super()` target | Always direct parent | Next in MRO (may skip parent!) |
| Type check | `obj instanceof Dog` | `isinstance(obj, Dog)` |
| Exact type check | N/A | `type(obj) == Dog` (no inheritance!) |
| Multi-type check | Chained `instanceof` | `isinstance(obj, (Dog, Cat))` |
| Class check | `Dog.class.isAssignableFrom(...)` | `issubclass(Dog, Animal)` |
| Hook into subclassing | Not possible | `__init_subclass__(**kwargs)` |
| Prevent subclassing | `final class Foo` | `@final` (type-checker) or `__init_subclass__` |
| Prevent overriding | `final void method()` | `@final` (type-checker only) |